# 03 — Federated K-Means Runs

Run federated k-means via FeatureCloud and aggregate results.

**What this notebook does:**
1. Prepare per-site input directories for the FeatureCloud fc_kmeans app.
2. Launch federated k-means tests (requires Docker + FeatureCloud CLI).
3. OR aggregate existing federated outputs (if FeatureCloud is unavailable).
4. Join federated cluster assignments with metadata.

**Prerequisites:**
- Run notebook 01 first (data preparation).
- For live runs: Docker running + FeatureCloud CLI installed.
- For aggregation only: existing zip results in `real_datasets/<dataset>/inputs/*/tests/`.

## Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    discover_clients,
    load_feature_matrix,
    load_metadata,
    prepare_variant_inputs,
    aggregate_fed_clusters,
)
from evaluation_utils.featurecloud_kmeans_utils import (
    ensure_app_image,
    run_single_federated_variant,
    aggregate_existing_federated_output,
)

## Configuration

In [2]:
DATASETS = ["ecoli", "ovarian_cancer", "quartet", "ccRCC_proteomics", "multiomics"]

# Set to True to launch a real FeatureCloud test.
# Set to False to only aggregate existing zip results.
RERUN_FEDERATED = True

# FeatureCloud settings (only needed when RERUN_FEDERATED=True)
APP_IMAGE = "fc_kmeans_upd"
APP_SOURCE_DIR = NOTEBOOK_DIR.parent / "federated_kmeans_upd"
CONTROLLER_HOST = "http://localhost:8000"
QUERY_INTERVAL = 5
TIMEOUT = 1800

OUTPUT_ROOT = NOTEBOOK_DIR

## Prepare FeatureCloud Inputs

Build per-site input directories with `intensities.tsv`, `design.tsv`,
and `config_kmeans.yml` — the format required by the fc_kmeans app.

This step creates `real_datasets/<dataset>/inputs/{before,corrected}/<site>/`
from the prepared matrices saved by notebook 01. It always runs (fast and idempotent)
so that the federated step below has up-to-date inputs.

In [3]:
# Always generate per-site input directories (fast, idempotent).
# These are needed for both fresh federated runs and for aggregating existing results.
configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    prepared_dir = ds_dir / "prepared"

    # Load prepared data from notebook 01
    before = load_feature_matrix(prepared_dir / "before_matrix.tsv")
    corrected = load_feature_matrix(prepared_dir / "corrected_matrix.tsv")
    metadata = pd.read_csv(prepared_dir / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: preparing FC inputs for k={k_values}")
    print(f"{'='*60}")

    # Generate per-site input directories for the fc_kmeans app
    before_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="before",
        matrix=before, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    corrected_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="corrected",
        matrix=corrected, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    print(f"Created: {before_dir}")
    print(f"Created: {corrected_dir}")


ecoli: preparing FC inputs for k=[2, 5]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ecoli/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ecoli/inputs/corrected

ovarian_cancer: preparing FC inputs for k=[2, 6]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ovarian_cancer/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ovarian_cancer/inputs/corrected

quartet: preparing FC inputs for k=[4]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/quartet/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/quartet/inputs/corrected

ccRCC_proteomics: preparing FC inputs for k=[2, 3]
Created: /home/yul

## Run or Aggregate Federated K-Means

When `RERUN_FEDERATED=True`, this launches a FeatureCloud test for each variant.
When `False`, it aggregates existing zip results from previous runs.

In [4]:
for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    metadata = pd.read_csv(ds_dir / "prepared" / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)
    client_names = [c.name for c in clients]

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: {'running' if RERUN_FEDERATED else 'aggregating'} federated k-means")
    print(f"{'='*60}")

    for variant in ["before", "corrected"]:
        variant_input_dir = ds_dir / "inputs" / variant

        try:
            output_path = run_single_federated_variant(
                dataset_name=ds_name,
                variant_label=variant,
                variant_input_dir=variant_input_dir,
                dataset_root=ds_dir,
                metadata=metadata,
                client_names=client_names,
                k_values=k_values,
                app_image=APP_IMAGE,
                controller_host=CONTROLLER_HOST,
                query_interval=QUERY_INTERVAL,
                timeout=TIMEOUT,
                keep_extracted=False,
                aggregate_only=not RERUN_FEDERATED,
            )
            print(f"  {variant}: saved to {output_path}")
        except FileNotFoundError as e:
            print(f"  {variant}: SKIPPED — {e}")
        except RuntimeError as e:
            print(f"  {variant}: FAILED — {e}")


ecoli: running federated k-means
[ecoli] Starting FeatureCloud controller for 'before' data
Killing leftover containers...
Starting the FeatureCloud controller with the data directory /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ecoli/inputs/before


Downloading...: 3it [00:00, 2693.84it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ecoli] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_341245380', 'containerID': '568188bd4d1b3bdc', 'coordinator': True, 'frontendUrl': '/app-traffic/3df90c4cb5/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_966800479', 'containerID': '170e4dfdc44e5feb', 'coordinator': False, 'frontendUrl': '/app-traffic/efd0ab9afe/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_635000138', 'containerID': 'a03f27f908bb99a5', 'coordinator': False, 'frontendUrl': '/app-traffic/e5f02ee14d/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_330682111', 'containerID': '8f1862a6ce632c89', 'coordinator': False, 'frontendUrl': '/app-traffic/8952c14658/', 'mess

Downloading...: 3it [00:00, 2290.72it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ecoli] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_966967576', 'containerID': '483c3570a5d15904', 'coordinator': False, 'frontendUrl': '/app-traffic/c762dd3aa9/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_826527520', 'containerID': 'eb749c478ced145c', 'coordinator': True, 'frontendUrl': '/app-traffic/28274b75ea/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_650843935', 'containerID': 'db31caa50ba50968', 'coordinator': False, 'frontendUrl': '/app-traffic/c9ef98203c/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_518128714', 'containerID': 'a3b78fc4bfbe12e2', 'coordinator': False, 'frontendUrl': '/app-traffic/151cbd81e4/', 'm

...

Controller started successfully, waiting 5s to ensure the controller is properly running!
[ccRCC_proteomics] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_921729179', 'containerID': '50baedc1ad19146e', 'coordinator': False, 'frontendUrl': '/app-traffic/16672992e5/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_467864746', 'containerID': 'd7356df1beeb3a93', 'coordinator': True, 'frontendUrl': '/app-traffic/0df3eb2619/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_9811774', 'containerID': '3841b757f20b2144', 'coordinator': False, 'frontendUrl': '/app-traffic/47563a3269/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[ccRCC_proteomics] Federated output saved: /home/yuliya-cosybio/r

Downloading...: 3it [00:00, 2203.66it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[multiomics] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_70650810', 'containerID': '18757f27780ec49d', 'coordinator': False, 'frontendUrl': '/app-traffic/21d11dc986/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_597289312', 'containerID': 'da707ade9d32335a', 'coordinator': True, 'frontendUrl': '/app-traffic/d8f875c506/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_125657010', 'containerID': '684ff8a128977ada', 'coordinator': False, 'frontendUrl': '/app-traffic/1b44412f79/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[multiomics] Federated output saved: /home/yuliya-cosybio/repos/cosybio/f

Downloading...: 3it [00:00, 2389.01it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[multiomics] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_603234148', 'containerID': '7e56aee9ca2aaf3d', 'coordinator': False, 'frontendUrl': '/app-traffic/3a21136fcf/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_154848841', 'containerID': 'ed2b0b1cffcf5614', 'coordinator': True, 'frontendUrl': '/app-traffic/c4a8b69ded/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_701380451', 'containerID': 'b8168c5170be5013', 'coordinator': False, 'frontendUrl': '/app-traffic/2310781559/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[multiomics] Federated output saved: /home/yuliya-cosybio/repos/cosyb

## Verify Outputs

Check that federated metadata files were produced.

In [5]:
for ds_name in DATASETS:
    run_dir = OUTPUT_ROOT / ds_name / "kmeans_res" / "runs"
    for fname in ["1_metadata_before_fedclusters.tsv", "1_metadata_after_fedclusters.tsv"]:
        p = run_dir / fname
        if p.exists():
            df = pd.read_csv(p, sep="\t")
            print(f"{ds_name}/{fname}: {df.shape[0]} samples, cols={list(df.columns)}")
        else:
            print(f"{ds_name}/{fname}: NOT FOUND")

ecoli/1_metadata_before_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
ecoli/1_metadata_after_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
ovarian_cancer/1_metadata_before_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
ovarian_cancer/1_metadata_after_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
quartet/1_metadata_before_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
quartet/1_metadata_after_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
ccRCC_proteomics/1_metadata_before_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3clusters']
ccRCC_proteomics/1_metadata_after_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3clusters']
multiomics/1_metadat